# Thai Election OCR — PaddleOCR-VL Pipeline

Extract structured voting data from scanned Thai election documents (Form สส.6/1).

**Metric:** Mean Levenshtein Distance (lower = better, 0 = perfect)

In [ ]:
# ============================================================
# Cell 1: Fix PDX + Imports (ALL IN ONE CELL - critical!)
# ============================================================
import os, sys

# Step A: Set env var BEFORE any paddlex import
os.environ['PDX_EAGER_INITIALIZATION'] = 'false'

# Step B: If paddlex was already loaded, monkey-patch to prevent re-init crash
if 'paddlex' in sys.modules:
    import paddlex
    paddlex._initialize = lambda: None  # make it a no-op
    print('[*] Patched paddlex._initialize (already loaded)')

# Step C: If paddlex.repo_manager.core exists, patch initialize() too
if 'paddlex.repo_manager.core' in sys.modules:
    import paddlex.repo_manager.core as _pdx_core
    _pdx_core.initialize = lambda *a, **k: None
    print('[*] Patched paddlex.repo_manager.core.initialize')

# Step D: Now safe to import paddleocr
import glob
import json
import re
import pandas as pd
from tqdm.auto import tqdm
from difflib import SequenceMatcher
from collections import defaultdict
from pathlib import Path

from paddleocr import PaddleOCRVL

# Paths
DATA_DIR = 'data'
IMAGES_DIR = os.path.join(DATA_DIR, 'images')
SAMPLE_LABELS_DIR = os.path.join(DATA_DIR, 'sample_labels')
TEMPLATE_PATH = os.path.join(DATA_DIR, 'submission_template.csv')
OUTPUT_SUBMISSION = 'submission.csv'
JSON_OUTPUT_DIR = 'ocr_output_json'
MD_OUTPUT_DIR   = 'ocr_output_md'

os.makedirs(JSON_OUTPUT_DIR, exist_ok=True)
os.makedirs(MD_OUTPUT_DIR, exist_ok=True)

print('[+] All imports successful.')

In [ ]:
# ============================================================
# Cell 2: Initialize PaddleOCR-VL
# ============================================================
print('[*] Loading PaddleOCR-VL model...')
pipeline = PaddleOCRVL(pipeline_version='v1')
print('[+] PaddleOCR-VL model loaded successfully.')

In [ ]:
# ============================================================
# Cell 3: Discover & group images by document
# ============================================================
all_images = sorted(glob.glob(os.path.join(IMAGES_DIR, '*.png')))
print(f'[*] Total page images: {len(all_images)}')

doc_pages = defaultdict(list)
for img_path in all_images:
    fname = os.path.basename(img_path).replace('.png', '')
    base_doc = re.sub(r'_page\d+$', '', fname)
    doc_pages[base_doc].append(img_path)

print(f'[*] Unique documents: {len(doc_pages)}')
for k in list(doc_pages.keys())[:3]:
    print(f'  {k}: {[os.path.basename(p) for p in doc_pages[k]]}')

In [ ]:
# ============================================================
# Cell 4: Run PaddleOCR-VL on all images (with resume)
# ============================================================
processed = set()
for jf in glob.glob(os.path.join(JSON_OUTPUT_DIR, '**', '*.json'), recursive=True):
    processed.add(os.path.basename(os.path.dirname(jf)))

remaining = [p for p in all_images if os.path.basename(p).replace('.png', '') not in processed]
print(f'[*] Already processed: {len(processed)}, Remaining: {len(remaining)}')

if remaining:
    for img_path in tqdm(remaining, desc='OCR Processing'):
        fname = os.path.basename(img_path).replace('.png', '')
        json_save = os.path.join(JSON_OUTPUT_DIR, fname)
        md_save   = os.path.join(MD_OUTPUT_DIR, fname)
        os.makedirs(json_save, exist_ok=True)
        os.makedirs(md_save, exist_ok=True)
        try:
            outputs = pipeline.predict(img_path)
            for res in outputs:
                res.save_to_json(save_path=json_save)
                res.save_to_markdown(save_path=md_save)
        except Exception as e:
            print(f'  [-] Error on {fname}: {e}')

print('[+] OCR extraction complete.')

In [ ]:
# ============================================================
# Cell 5: Inspect sample outputs
# ============================================================
# JSON
sample_jsons = glob.glob(os.path.join(JSON_OUTPUT_DIR, '**', '*.json'), recursive=True)
if sample_jsons:
    with open(sample_jsons[0], 'r', encoding='utf-8') as f:
        sample_data = json.load(f)
    print(f'=== JSON: {sample_jsons[0]} ===')
    if isinstance(sample_data, dict):
        print('Keys:', list(sample_data.keys()))
        if 'parsing_res_list' in sample_data:
            for i, block in enumerate(sample_data['parsing_res_list'][:5]):
                print(f"\n  Block {i} [{block.get('block_label','')}]:")
                print(f"    {str(block.get('block_content',''))[:300]}")
        else:
            print(json.dumps(sample_data, ensure_ascii=False, indent=2)[:2000])

# Markdown
print('\n' + '='*60)
sample_mds = glob.glob(os.path.join(MD_OUTPUT_DIR, '**', '*.md'), recursive=True)
if sample_mds:
    with open(sample_mds[0], 'r', encoding='utf-8') as f:
        print(f'=== MD: {sample_mds[0]} ===')
        print(f.read()[:3000])

In [ ]:
# ============================================================
# Cell 6: Helper functions
# ============================================================

def normalize_votes(raw):
    """Convert any OCR vote string to clean Arabic digit string."""
    s = str(raw).translate(str.maketrans('๐๑๒๓๔๕๖๗๘๙', '0123456789'))
    s = s.replace(',', '').replace(' ', '').replace('.', '')
    s = re.sub(r'[^0-9]', '', s)
    return str(int(s)) if s else '0'


def fuzzy_match_party(target, candidates, threshold=0.55):
    """Return best fuzzy match from candidates or None."""
    best, best_r = None, 0.0
    for c in candidates:
        r = SequenceMatcher(None, target, c).ratio()
        if r > best_r:
            best, best_r = c, r
    return best if best_r >= threshold else None


def extract_party_votes_from_md(md_text):
    """Extract party-vote pairs from markdown text."""
    results, seen = [], set()
    lines = md_text.split('\n')
    
    # Strategy 1: Markdown table rows
    for line in lines:
        line = line.strip()
        if '|' not in line or re.match(r'^[\\|\\s\\-:]+$', line):
            continue
        cells = [c.strip() for c in line.split('|') if c.strip()]
        if len(cells) < 2:
            continue
        
        thai_cells, num_cells = [], []
        for idx, cell in enumerate(cells):
            cleaned = normalize_votes(cell)
            if re.search(r'[\u0E00-\u0E7F]', cell):
                thai_cells.append((idx, cell))
            if re.match(r'^\d+$', cleaned) and cleaned != '0':
                num_cells.append((idx, cleaned))
        
        for t_idx, t_cell in thai_cells:
            party = re.sub(r'^[\d\.\s]+', '', t_cell).strip()
            party = re.sub(r'[\d\.]+$', '', party).strip()
            if not party or len(party) < 2:
                continue
            for n_idx, n_val in num_cells:
                key = f'{party}_{n_val}'
                if key not in seen:
                    results.append({'party': party, 'votes': n_val})
                    seen.add(key)
                    break
    
    # Strategy 2: Free-form text
    if not results:
        for line in lines:
            for party_raw, votes_raw in re.findall(
                r'([\u0E00-\u0E7F][\u0E00-\u0E7F\s]{1,30}?)\s+([\d๐-๙][\d๐-๙,\.\s]{0,10})', line
            ):
                party, votes = party_raw.strip(), normalize_votes(votes_raw)
                key = f'{party}_{votes}'
                if party and votes != '0' and key not in seen:
                    results.append({'party': party, 'votes': votes})
                    seen.add(key)
    return results


def extract_party_votes_from_json(json_data):
    """Extract party-vote pairs from PaddleOCR-VL JSON output."""
    results = []
    if isinstance(json_data, dict) and 'parsing_res_list' in json_data:
        for block in json_data['parsing_res_list']:
            content = block.get('block_content', '')
            if content:
                results.extend(extract_party_votes_from_md(content))
    if not results and isinstance(json_data, dict):
        results = extract_party_votes_from_md(json.dumps(json_data, ensure_ascii=False))
    return results


print('[+] Helpers defined.')

In [ ]:
# ============================================================
# Cell 7: Build vote mapping from OCR outputs
# ============================================================
vote_mapping = defaultdict(dict)

json_files = glob.glob(os.path.join(JSON_OUTPUT_DIR, '**', '*.json'), recursive=True)
print(f'[*] JSON files: {len(json_files)}')

for jf in tqdm(json_files, desc='Parsing JSON'):
    parent = os.path.basename(os.path.dirname(jf))
    doc_id = re.sub(r'_page\d+$', '', parent)
    try:
        with open(jf, 'r', encoding='utf-8') as f:
            data = json.load(f)
        for p in extract_party_votes_from_json(data):
            party, votes = p['party'], p['votes']
            if party not in vote_mapping[doc_id]:
                vote_mapping[doc_id][party] = votes
            else:
                old = int(vote_mapping[doc_id][party])
                vote_mapping[doc_id][party] = str(max(old, int(votes)))
    except Exception:
        continue

# Fallback: Markdown
md_files = glob.glob(os.path.join(MD_OUTPUT_DIR, '**', '*.md'), recursive=True)
for mf in tqdm(md_files, desc='Parsing MD (fallback)'):
    parent = os.path.basename(os.path.dirname(mf))
    doc_id = re.sub(r'_page\d+$', '', parent)
    if doc_id in vote_mapping and vote_mapping[doc_id]:
        continue
    try:
        with open(mf, 'r', encoding='utf-8') as f:
            for p in extract_party_votes_from_md(f.read()):
                vote_mapping[doc_id][p['party']] = p['votes']
    except Exception:
        continue

print(f'\n[+] Mapped {len(vote_mapping)} documents.')
for doc in list(vote_mapping.keys())[:2]:
    print(f'  {doc}: {dict(list(vote_mapping[doc].items())[:4])}')

In [ ]:
# ============================================================
# Cell 8: Validate against sample labels
# ============================================================

def levenshtein(s1, s2):
    if len(s1) < len(s2): return levenshtein(s2, s1)
    if not s2: return len(s1)
    prev = range(len(s2) + 1)
    for i, c1 in enumerate(s1):
        curr = [i + 1]
        for j, c2 in enumerate(s2):
            curr.append(min(prev[j+1]+1, curr[j]+1, prev[j]+(c1!=c2)))
        prev = curr
    return prev[-1]

def resolve_vote_from_mapping(doc_id, party, mapping):
    ext = mapping.get(doc_id, {})
    if not ext: return '0'
    if party in ext: return ext[party]
    for ep, ev in ext.items():
        if party in ep or ep in party: return ev
    m = fuzzy_match_party(party, ext.keys())
    return ext[m] if m else '0'

sample_labels = glob.glob(os.path.join(SAMPLE_LABELS_DIR, '*.json'))
print(f'[*] Validating {len(sample_labels)} samples...')
total_dist, total_count, misses = 0, 0, []

for sl in sample_labels:
    with open(sl, 'r', encoding='utf-8') as f:
        label = json.load(f)
    doc_id = f"{label['type']}_{label['province_code']}_{label['constituency_number']}"
    for r in label['results']:
        true_v = str(r['votes'])
        pred_v = resolve_vote_from_mapping(doc_id, r['party'], vote_mapping)
        d = levenshtein(pred_v, true_v)
        total_dist += d; total_count += 1
        if d > 0:
            misses.append({'doc': doc_id, 'party': r['party'], 'true': true_v, 'pred': pred_v, 'dist': d})

if total_count:
    print(f'Mean Levenshtein: {total_dist/total_count:.4f}')
    print(f'Perfect: {total_count-len(misses)}/{total_count} ({100*(total_count-len(misses))/total_count:.1f}%)')
    for m in sorted(misses, key=lambda x: -x['dist'])[:10]:
        print(f"  {m['doc']}|{m['party']}: true={m['true']} pred={m['pred']} dist={m['dist']}")

In [ ]:
# ============================================================
# Cell 9: Generate submission.csv
# ============================================================
sub_df = pd.read_csv(TEMPLATE_PATH)
print(f'[*] Template: {len(sub_df)} rows')

def resolve_vote(row):
    return resolve_vote_from_mapping(
        str(row['doc_id']), str(row.get('party_name','')).strip(), vote_mapping
    )

tqdm.pandas(desc='Resolving')
sub_df['votes'] = sub_df.progress_apply(resolve_vote, axis=1).astype(str)

non_zero = (sub_df['votes'] != '0').sum()
print(f'[+] Non-zero: {non_zero}/{len(sub_df)} ({100*non_zero/len(sub_df):.1f}%)')
sub_df.to_csv(OUTPUT_SUBMISSION, index=False)
print(f'[+] Saved: {OUTPUT_SUBMISSION}')
sub_df.head(20)